<a href="https://colab.research.google.com/github/SotaYoshida/Lecture_DataScience/blob/master/Python_chapter5_handling_files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ファイル操作

[この章の目的]
text,csvやxlsx形式のデータをプログラムでサクッと扱えるようになる。

この章では、テキストファイルやcsvファイル(excelファイル)をPythonで操作する簡単な方法を学習します。

これまでの章では、データはリストとして既に与えられた状態から解析を行ったが、実際にデータを扱う際は既に誰かが作成した何らかのファイルをプログラムで読み込んで操作する必要があります。
この章の内容は、データ解析というよりは、Pythonでデータ解析をするための下準備や前処理と呼ばれる部分に相当します。

愚直にコードを書いている事も手伝って、少々泥臭い部分が多いですが、
この章のような操作のエッセンスを抑えておけば、普通にやると膨大な時間がかかる様々な処理を高速化することができると思うので、頑張りましょう。


## テキストファイルの操作

膨大な行のテキストファイルに対して、人間が手で操作をするというのは時として非現実的です。
誤変換を置換するくらいなら、どのテキスト/メモ帳アプリやwordでもできますが、
全行(数千とか数万)に対して、決まった操作が必要な場合、プログラムにしてしまったほうが遥かに便利です。

まずは
https://drive.google.com/file/d/1xhTpHVi3BanXJwF2GLzUEty17CEeFy1Q/view?usp=sharing
にアクセスして、テキストファイルを保存し、google driveの適当なフォルダにアップロードしてください。

以下では、google driveのマイドライブの```AdDs2020```というフォルダにファイルを保存したと仮定して話を進めますので、適宜皆さんの場合に置き換えて使用してください。

まずはgoogle driveに保存した```test.txt```という名前のファイルを読み込んでみましょう。
既に何回かやったようにgoogle driveをマウントします。




In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/My\ Drive/AdDS2020/* #ターミナルコマンドでspaceを表現するときはバックスラッシュを入れる

とするとgoogle driveのマイドライブ/AdDS2020にあるファイルの一覧を表示させることができます。

In [ ]:
!ls /content/drive/My\ Drive/AdDS2020/*txt ## /content/drive/AdDS2020以下の.txtファイルを全て表示
!ls /content/drive/"My Drive"/AdDS2020/*txt ##これでもOK

test.txtが見つかったでしょうか？ (アップロードしてから読み込みできるまでに少し時間がかかる場合があります)

では次に、このファイルに書かれているテキストを取得してみましょう。

方法はいくつかありますが、最もポピュラーなものとして、ファイルを開いてテキストを取得する方式を採用します。

In [ ]:
filename = "/content/drive/My Drive/AdDS2020/test.txt" 
inp = open(filename,"r") # "r"はread(読み出し)を意味するオプション
lines = inp.readlines()
inp.close()

1行目でファイル名(正確にはファイルのパス)を指定し、```filename```という名前に格納しました。  
2行目では、指定したパスにあるファイルを開いています。今はファイルに書き込むのではなく、既にあるファイルを開いて読み込むので、"r"というオプションを指定します。他には"w"(書き出し,上書き), "a"(書き出し,追記)などがあります。新しく上書きでファイルを作成したい場合は"w", すでにあるファイルの内容は消さずに追記したい場合は"a"を指定します。

3行目では、inpという名前をつけたファイルに対して、```readlines```という操作を適用しています。
これは、ファイルに書かれているテキストを[全行に渡って]読み込む関数です。
実際、


In [ ]:
print(lines)

とすると、全ての行が読み込まれ、変数```lines```に格納されていることがわかります。

ここで```\n```は改行記号を意味します。
ループを回して一行ずつ表示させると

In [ ]:
for line in lines:
    print(line)

といった感じです。(今は改行コードを含んでprintしているので一行ずつ間が空いています)

特定の1行しか必要がなければ、複数形ではなく```readline()```としてforループを回しても構いませんし、必要な行番号が分かっている場合は、nlines = lines[2:10]などとして要らないところは捨てて解析・操作してもOKです。

もう少し具体的なテキスト操作をしてみましょう。

まず、上の1行ずつ表示するコードでは、改行コードを明示的に含む文字列を一行ずつ表示したため、余分なスペースが空いてしまいました。
たとえば```strip()```関数を使うことで改行コードを消去することができます。

In [ ]:
for line in lines:
    print(line.strip())

右側に改行コードが入っていることが明確な場合はstripの代わりにrstripを使ってもOKです(rstripのrはrightの意味)。

In [ ]:
for line in lines:
    print(line.rstrip())

としても同じ結果が得られます。ファイルによってはインデントをするために左側にタブ```\t```が含まれる場合もあります。
そのような場合には```strip()```や```lstrip()```を使って取り除くことができます。

たとえば#記号を含む行以降だけが必要な場合はどうすればいいでしょうか？
最も単純な実装は

In [ ]:
hit = 0 #
for line in lines:
    if "###" in line:
        hit += 1 
        continue
    if hit == 0 :
        continue #hitが0の状態では何もしない
    print(line.rstrip())

といったところでしょうか。

以下では、###dataまでの行が必要ないので、必要なところまでを変数に格納してしまいましょう。



In [ ]:
hit = 0 #
nlines = []
for line in lines:
    if "###" in line:
        hit += 1 
        continue
    if hit == 0 :
        continue #hitが0の状態では何もしない
    nlines += [line]
print(nlines)

また、1,2,3,4,5,6といったコンマやスペースで区切られたものをリストに格納したい場合には、```split```関数が便利です。

引数に何もしていしなければ、スペースもしくはタブごとに文を区切ったリストを返します。

In [ ]:
for line in nlines:
    print(line.split())

カンマがあるときはカンマで分割する、という約束を表現したければ

In [ ]:
for line in nlines:
    if "," in line :
        print(line.rstrip().split(","))
    else :
        print(line.rstrip().split())

などとすることができます。たとえばこれを利用すれば、
空のリストにファイルから読んだ要素を詰めていくといった操作が実現できます。


In [ ]:
nums = [] #数字のリストなのでnumsとした
profs = [] #プロフィールのリストなのでprodsとした

for line in nlines:
    if "," in line :
        nums += [ line.rstrip().split(",") ]
    else :
        profs += [ line.rstrip().split()]
print("nums", nums)
print("profs", profs)

上のnumsの様に、予め全ての要素が整数だと分かっていて、整数に対する演算(四則演算など)を後でするのなら、str(文字列)型ではなく、int型にしておくほうが良いでしょう。

In [ ]:
nums = []
for line in nlines:
    if "," in line :
        nums += [ list(map(int,line.rstrip().split(","))) ]
print(nums)

map関数はmap(操作,対象)という風に使って、対象の各要素に対して操作を適用することができます。
今の場合、['1', ' 2', ' 3', ' 4', ' 5', ' 6']などの文字列のリストに対して、整数型に変換する```int```関数を作用させるという操作を一度に行います。

注: map関数の返り値はmap objectと呼ばれるものなので、元のようなリストの形で使いたい場合はlist()を使ってリストに変換するステップが必要です。

世の中には、アンケート結果や産業データなどがcsv(カンマ区切りのテキスト)ファイルで公開されている場合が多いですが、その場合は上で説明したような手順でリストなどに格納すれば、今まで行ったような解析が実行できる、といったわけです。

テキストファイルを書き込んで保存する、というのもやってみましょう。

上の文字列で、敬称を"さん"から"様"に置換したテキストを保存してみましょう。


In [ ]:
filename = "/content/drive/My Drive/AdDS2020/test_replace.txt" 
oup = open(filename,"w")  ## oup は"output"の気持ち...
for line in lines:
    print(line.rstrip().replace("さん:","様:"), file=oup) # file=[openしたファイル]にすることで、printする先をファイルに指定できます。
oup.close() #ファイルはきちんと閉じる.

## エクセルファイルの操作

### アンケート分析

https://drive.google.com/file/d/1bYJNWdtujcQWfSBAa1UeXi2ZzJRJktil/view?usp=sharing
に、あるアンケート結果をまとめたファイルを公開しました。

これは、Google フォームで作成したhttps://docs.google.com/forms/d/1XU5yEoLzd-Jvcba7GaHhMGAZDzDd_Cpl0Jylvullc28/edit?usp=sharing
で、性別、国数英社理(中学の５科目)に対する得意/苦手意識の調査を想定した疑似アンケートです。

> 余談: このcsvファイルをExcelで開こうとすると文字化けを起こす。これはgoogleフォームで作成されたcsvファイルの文字コードが世界標準のutf-8を使用しているのに対し、ExcelがShift-JISという時代遅れな文字コードでcsvファイルを開こうとするため。
Googleのスプレッドシートや、Macに標準で搭載されているNumbersで開くと文字化けしません。

> 2000件の回答は、もちろん私が手作業で入力したわけでも誰かに協力してもらったわけでもなく、一定のルール(傾向)を勝手に設定してランダムに回答を作成しフォームから自動回答するPythonスクリプトを書きました。時間に余裕があれば、こうしたWeb操作を自動化する方法も授業で扱います。 c.f. ブラウザ操作, Webスクレイピング

このようなアンケート調査は事務作業や卒業研究などで頻繁に見られ、会社や大学、所属しているコミュニティなどで何らかの意思決定に用いられることも多いことでしょう。

こうしたアンケート分析を行っていると、

*   各回答項目同士の関係が知りたい
*   設定していなかった項目の情報も抽出したい

といった要望が出てきます。

今の場合でいうと、
* 各科目ごとの得意・苦手意識の相関を調べたい
* 夜中(あるいは日中)にアンケートを回答した夜型(昼型)の人に見られる特徴がなにかないか？

といったイメージです。そんなとき、

> 国語が得意(どちらかというと得意)と回答した方に質問です。　英語についてはどうでしょうか？

などと新たに設問を増やしてアンケートをやり直すというのは得策では有りません。

すでに得られた情報から、さらなる情報をPythonを使ってサクッと引き出すことを考えてみましょう。

まずは、csvファイルに記載された情報を整理してプログラムで扱いやすくすることを考えます。


In [ ]:
filename = "/content/drive/My Drive/AdDS2020/python_handling_test.csv" #読み込むファイルのパスの指定

とりあえずファイルの中身を数行表示してみる。

csvファイル(コンマ区切りのテキスト)なので、テキストファイルと同じ方法をとる(他の方法ももちろんある)

In [ ]:
inp=open(filename,"r")
lines=inp.readlines()
inp.close()
print("行数は",len(lines))
for i in range(5):
    print(lines[i].rstrip())

> ちなみに...```pandas```ライブラリを使うと

In [ ]:
import pandas as pd 
df = pd.read_csv(filename)
print(df)

さて、```lines```に格納したデータをもう少し扱いやすいように変更しよう。  
最初の0行目はどういうデータが入っているか(データの項目)を表している。
1-2000行目には2000人分の回答が詰まっている。  
(*Pythonでのインデックス指定と整合するように０からカウントすることにした)

これによると、  
> 0列目: 回答した時刻  
> 1列目: 性別  
> 2列目: 国語  
> 3列目: 数学  
> 4列目: 英語  
> 5列目: 社会  
> 6列目: 理科  

らしい。いろいろなデータの整理方法があると思うがここでは、
* A) 0列目の時刻を２４時間表記にする  
* B) 2-6列目の各科目の得意・苦手意識を、文字列を除去して数値[-2,-1,0,1,2]として扱う

をまず実行する






In [ ]:
#処理Aのための関数
#input_strが、"年月日 時刻(h:m:s) 午前/午後 GMT+9" という文字列である、という文字列の[構造]を知った上での実装になっていることに注意
def make_time_24h(input_str):    
    time  = input_str.split()[1]
    hms = time.split(":")
    h = int(hms[0])
    if time[2] == "午前":
        output_str = time 
    else :
        if h != 12:
            output_str = str(h +12)+":"+hms[1]+":"+hms[2]
        else:
            output_str = str(h)+":"+hms[1]+":"+hms[2] # 12時xx分だけは別の取り扱いが必要
    return output_str


nlines=[] #整理したものをリストとしてまとめるための空のリスト
for nth,line in enumerate(lines[1:]): 
    nline = line.rstrip().replace('"','').split(",") # 改行文字の除去、ダブルクォーテーションの除去, カンマで分割    
    # この時点でnlineは0:時刻 1:性別, ...のリストとなっているはず print()でcheckしてみよう
    # 処理A)
    time = make_time_24h(nline[0])
    M_or_F = nline[1] #性別

    #　処理B)
    points = [ int(nline[k].split()[0]) for k in range(2,7)] #各科目の値だけのリスト(points)を作成
    nline = [time, M_or_F]+points  #リストを連結(時刻,性別と各科目の値を同じ階層で結合)して、nlineという名前で上書き
    nlines += [ nline ]

    # うまく編集できたか400行おきほどでprintしてチェックしてみる
    if nth % 400 == 0 :
        print("編集前", line.rstrip())
        print("編集後", nline)
        print("")

最後に、各項目の得点を適当なリスト(あるいはnp.array)に整形しておけば、種々の分析を行うことができます。



In [ ]:
import numpy as np
points = [ [] for i in range(5)]
for tmp in nlines:
    for i in range(5):
        points[i]+=[tmp[2+i]]
print("points", np.array(points))
print("各科目の平均スコア:", [np.mean(points[i]) for i in range(5)])

相関分析は以降の章で扱うので具体例は省略します。


## おまけ

以降では、2020年度前期のデータサイエンス入門(一部学科を除く)の相関分析で使用されたエクセルファイルを使用します。  
下記のリンクよりファイル(kakei.xlsx)をダウンロードして、google driveの適当な箇所にアップロードしてください。  
以下では、google driveのマイドライブ直下のAdDs2020というフォルダにファイルを置いたと思って処理を行いますので、適宜ご自身の環境に置き換えてください。



In [ ]:
filename = "/content/drive/My Drive/AdDS2020/kakei.xlsx" #読み込むファイルのパスの指定

まずはxlsxファイルをPythonで読み込んで、どんな"シート"があるのかを確認してみましょう。

In [ ]:
import xlrd #excel readモジュールをimportする
wb = xlrd.open_workbook(filename) #作業するbook(excelファイル)を開いてwbという変数名をつける
sheets = wb.sheets # ブック内のシートを取得(.sheets)して、sheetsという変数名をつける
print("シート名のリスト", wb.sheet_names())

たくさんシートがあることが分かります。
このエクセルを実際に開くと、Sheet1からSheet12までが複数都市の家計調査のデータで、S1からS12までが気候データになっていて、1-12までの数字が2017年の1月から12月までに対応していることが分かります。

このように、エクセルシートの[構造]を予め知っておけば、やりたい操作を系統的に実行することができます。



以下のコードは、プログラミングの"ありがたみ"を感じてもらうためのお試し用です。   
(昔書いたコードなのですが、かなり読みにくいコードなのであまり真剣に読まないでください)

大量の画像ファイルをドライブに生成するので、以下を読んだ上で実行してください。
何もいじらずに実行すると、全都市の月別平均気温と全品目の世帯平均支出のうち、相関係数の絶対値が0.9以上のものをプロットして画像として保存します。この条件をみたす項目は291通りあります。

Google Colab上で実行して画像が生成されるまでに110秒程度かかりました(私の手元のPCでは数十秒-100秒くらい).  
この時間未満でエクセルで操作をして同様の処理を完了出来るという方は...おそらく地球上にいないでしょう(要出典)

In [ ]:
!mkdir /content/drive/My\ Drive/AdDS2020/kakei_cor_pic # 画像がいっぱい生成されると面倒なので画像を保存するフォルダを作成しておく

In [ ]:
!pip install japanize_matplotlib #japanize_matplotlibモジュールのインストール. 一度実行すれば良い

In [ ]:
import xlrd
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import japanize_matplotlib
import time

class ebook:
    def __init__(self,inpf):
        self.inpf = inpf
        self.wb = xlrd.open_workbook(inpf)
        self.sheets = self.wb.sheets
        self.ns,self.sname = self.wb.nsheets, self.wb.sheet_names()
        s_kikou=[]; s_kakei=[]
        for i, sheetname in enumerate(self.sname):
            if "Sheet" in sheetname :
                s_kakei += [ i ]
            elif "S" in sheetname :
                s_kikou += [ i ]
        self.s_kakei,self.s_kikou = s_kakei,s_kikou
    def indices(self):
        return self.s_kakei, self.s_kikou
    def readkakei(self,ikakei) :
        ws = self.wb.sheet_by_index(ikakei)
        nr = ws.nrows; nc=ws.ncols
        premode = True
        items = []
        for ii in range(nr): 
            trow = ws.row_values(ii)
            hit = 0
            if premode == True:
                for jj,tmp in enumerate(trow):
                    try :
                        if "市" in tmp:
                            hit += 1
                    except:
                        hit = hit
                if hit > 5:
                    premode=False
                    i_kakei=[];p_kakei=[]
                    for jj,tmp in enumerate(trow):
                        if "市" in tmp:
                            i_kakei += [jj]
                            p_kakei +=[ tmp ] 
                    v_kakei = [ ]
            else:                    
                if ii >= 22:
                    if type(trow[8]) is str and trow[8] != "":
                        v_kakei += [ [trow[jj+1] for jj in i_kakei] ]
                        items += [trow[8]]                         
        return i_kakei, p_kakei, v_kakei,items
    def readkikou(self,ikikou):
        ws = self.wb.sheet_by_index(ikikou)
        nr = ws.nrows; nc=ws.ncols
        quantities = [];v_kikou=[]
        premode=True
        for ii in range(nr): 
            trow = ws.row_values(ii)
            if premode :
                if any(["市" in tmp for tmp in trow]):
                    Tplaces = trow[1:]
                    premode=False
            else:
                quantities += [ trow[0] ]
                v_kikou += [ trow[1:] ]
        return Tplaces, v_kikou,quantities

def seasoncolor(month):
    if month <= 2 or month ==12:
        return "blue"
    elif 3 <= month <=5:
        return "green"
    elif 6 <= month <=8:
        return "red"
    elif 9<= month <=11:
        return "orange"
    return tcol

def plot_cor(x,y,item,quantity,place,corrcoef):    
    fig = plt.figure(figsize=(4,4))
    ax = fig.add_subplot(1,1,1)
    ax.set_facecolor("#D3DEF1")
    ax.set_title(place+"   r="+str("%8.2f" % corrcoef).strip())
    ax.set_xlabel(item);ax.set_ylabel(quantity)
    ax.grid(True,axis="both",color="w", linestyle="dotted", linewidth=0.8)
    for i in range(len(x)):
        tcol=seasoncolor(i+1)
        ax.scatter(x[i],y[i],marker="o",s=5,color=tcol,zorder=20000,alpha=0.7)
        ax.text(x[i],y[i],str(i+1)+"月",color="k",fontsize=8)
    plt.savefig(oupdir + "corr_"+item+"vs"+quantity+"_at_"+place+".png",dpi=300) 
    plt.close()

def calcor(date,places,items, Vs_kakei,Tplaces,quantities,Vs_kikou):
    hit = 0
    Vs = [] 
    for j_K,place in enumerate(places):
        for j_T, Tplace in enumerate(Tplaces):
            if place != Tplace :
                continue
            for ik,item in enumerate(items):
                kvalue = np.array([ Vs_kakei[i][ik][j_K] for i in range(len(Vs_kakei))])
                quantity=quantities[iT]
                Tvalue = np.array([ Vs_kikou[i][iT][j_T] for i in range(len(Vs_kikou))])
                if all(Tvalue) == 0.0: ## missing value in climate data
                    continue
                if printlog:
                    print("@", place," ",item,kvalue," VS ",quantity, ",",Tvalue)
                corrcoef=np.corrcoef(kvalue,Tvalue)[0][1]
                Vs += [ [ corrcoef, item, quantity, place] ]
                if abs(corrcoef) > pthre:
                    if pltmode==True:
                        plot_cor(kvalue,Tvalue,item,quantity,place,corrcoef)
                        hit += 1
    print("hit=",hit)

if __name__ == "__main__":
    ti=time.time()

    T=True; F=False
    global oupdir #保存するディレクトリ名
    inpf = "/content/drive/My Drive/AdDS2020/kakei.xlsx"
    oupdir = "/content/drive/My Drive/AdDS2020/kakei_cor_pic/" #適宜置き換える
    global iT 
    iT = 6  # iT=6: 日平均気温

    printlog= F #条件にhitした都市の品目と気候データを逐次plotするかどうか Fを推奨
    pthre= 0.9 ## corrplotを描く相関係数のthreshold 0.9だけでも291個くらいグラフができるので
    pltmode = T ## T:plotする F:計算のみ ** 画像をいちいちplotして保存する必要がない場合、Fを推奨
    year="2017" 
    wb=ebook(inpf)
    s_kakei,s_kikou=wb.indices()   
    Vs_kakei=[]; Vs_kikou=[];dates=[]
    for i,ind_kakei in enumerate(s_kakei):
        i_places,places, v_kakei,items = wb.readkakei(ind_kakei)
        Tplaces, v_kikou, quantities  = wb.readkikou(s_kikou[i])
        if i+1 < 10:
            date=year+"0"+str(i+1)
        else:
            date=year+str(i+1)
        dates += [date]
        Vs_kakei += [ v_kakei ]
        Vs_kikou += [ v_kikou ]
    calcor(dates,places,items,Vs_kakei,Tplaces,quantities,Vs_kikou)    

    tf=time.time()
    print("Time[sec]:", tf-ti)